# Tutorial model run for the GeoEnthalpyDelta component for building a small delta fan

Builds a small delta prograding from a west-edge feeder onto a planar,
fixed basement, with a flat sea level, then plots map-view thickness and a
centerline cross-section (basement, topography, sea level).

The notebook runs the model first and explains the theory afterward: the
**Theory** section at the end walks back through what each step of the
code is doing.

This component implements the two-dimensional finite-volume core of the
enthalpy-based delta model described in Lorenzo-Trueba, Anderson, Bui, and
Voller, *GeoEnthalpy-Delta v1.0: an enthalpy-based model for coupled
subaerial and subaqueous delta evolution with diagnostic moving
boundaries* (manuscript in preparation for *Geoscientific Model
Development*).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid
from landlab.components import GeoEnthalpyDelta

In [ ]:
help(GeoEnthalpyDelta)

## Grid setup

The basement is the fixed, non-erodible surface $E(x,y)$ of Eq. (1) in the
paper, $E(x) = -\beta x$, with the basement slope $\beta$ absorbed into
the nondimensionalization so that the beta-normalized form used by the
code is simply $E(x) = -x$ (Sect. 4.3). $x$ increases down-system, from
the feeder toward the basin; $y$ is the lateral, cross-system direction.

Since there is no sediment yet, the initial `topographic__elevation` *is* the basement. The component works out the basement from it, and from then on only the sediment thickness on top of it evolves.

Because the component transports sediment at every node, we also mark every node, including the perimeter, as core (see
*Boundary conditions* in the **Theory** section).

In [ ]:
nrows, ncols = 50, 50
dx = dy = 0.2
# xy_spacing also accepts a (dx, dy) tuple if you need dx != dy; a scalar
# here just means dx == dy.
grid = RasterModelGrid((nrows, ncols), xy_spacing=dx)
grid.status_at_node[:] = grid.BC_NODE_IS_CORE

x = grid.x_of_node.reshape(grid.shape)
topo = grid.add_zeros("topographic__elevation", at="node")
topo[:] = (-x).reshape(-1)  # planar, non-erodible basement E(x) = -x
initial_topography = topo.copy()

## External forcing: sea level and sediment supply

The component reads two more fields that you own and can change between calls to `run_one_step`:

- `sea_level__elevation`, a single value stored *at grid*. Add to it inside the run loop for a rising (or falling) sea level. Here it stays fixed at $Z = -5$.
- `sediment__influx`, the volume of sediment per unit time entering each node. A feeder on the west edge is just influx on some nodes of the first column. Here we spread a total input rate $Q_{\mathrm{in}} = 0.5$ evenly over the rows within half a feeder width $w_f = 1$ of the grid's lateral center.

In [ ]:
sea_level = grid.add_field("sea_level__elevation", -5.0, at="grid")

sediment_flux = 0.5  # total input rate, Q_in
feeder_width = 1.0  # w_f

y_of_row = grid.y_of_node.reshape(grid.shape)[:, 0]
feeder_y_center = 0.5 * (y_of_row.min() + y_of_row.max())
feeder_rows = np.abs(y_of_row - feeder_y_center) <= 0.5 * feeder_width + 1.0e-12

influx = grid.add_zeros("sediment__influx", at="node")
influx.reshape(grid.shape)[feeder_rows, 0] = sediment_flux / np.count_nonzero(
    feeder_rows
)
print(f"feeder rows: {np.flatnonzero(feeder_rows)}, total influx = {influx.sum()}")

## Component setup

The keyword arguments below set the nondimensional, beta-normalized
parameters of Sect. 4.3 of the paper (the component is unit-agnostic
`_unit_agnostic = True` no physical units attach to any of them). Each
takes an `(x, y)` pair:

| keyword | paper symbols | meaning |
|---|---|---|
| `topset_threshold` | $(C_{\mathrm{top}}^x, C_{\mathrm{top}}^y)$ | critical slopes, topset |
| `foreset_threshold` | $(C_{\mathrm{fore}}^x, C_{\mathrm{fore}}^y)$ | critical slopes, foreset |
| `topset_diffusivity` | $(D_{\mathrm{top}}^x, D_{\mathrm{top}}^y)$ | transport coefficients, topset |
| `foreset_diffusivity` | $(D_{\mathrm{fore}}^x, D_{\mathrm{fore}}^y)$ | transport coefficients, foreset |

The **Theory** section at the end explains how these thresholds and
diffusivities enter the sediment flux.

The other quantities of the paper's setup are not keywords: the input rate
$Q_{\mathrm{in}}$ and feeder width $w_f$ live in the `sediment__influx` field,
and the sea level $Z(t)$ in the `sea_level__elevation` field (see the cell
above).

A finite (non-vertical) foreset requires $C_{\mathrm{fore}}^x$ to exceed
the beta-normalized basement slope of 1 (Sect. 5.2 of the paper); the
`foreset_threshold=(2.0, 2.0)` used below satisfies that, which is why the
deposit develops a genuine subaqueous toe rather than an abrupt drop-off.

In [ ]:
esd = GeoEnthalpyDelta(
    grid,
    topset_threshold=(0.25, 0.1),
    foreset_threshold=(2.0, 2.0),
    topset_diffusivity=(1.0, 1.0),
    foreset_diffusivity=(1.0, 1.0),
)

dt = 0.1  # run_one_step subcycles internally to stay CFL-stable
nsteps = 220
print(f"dt = {dt}, running {nsteps} steps (t_end = {nsteps * dt:.3g}) ...")

You should see `done in #.# s` printed once the run has completed.

In [ ]:
t0 = time.time()
volume_history = []
time_history = []
for _ in range(nsteps):
    esd.run_one_step(dt)
    volume_history.append(np.sum(esd.sediment_thickness) * dx * dy)
    time_history.append(esd.time_elapsed)
print(f"done in {time.time() - t0:.2f} s")

## Mass balance

The finite-volume update conserves sediment exactly except for the
`sediment__influx` input and the positivity clip at $H=0$, so the total stored volume

$$
V(t) = \sum_{i,j} H_{i,j}(t)\, \Delta x\, \Delta y
$$

should track the cumulative imposed input $V_{\mathrm{in}}(t) =
Q_{\mathrm{in}}\, t$ (Eqs. 56-58 of the paper) up to floating-point
roundoff. We use `esd.time_elapsed` for $t$, so the check stays valid even if you rerun the loop above.

If the fan reaches those edges before the run ends, sediment piles up against them and this check starts to fail; that's a sign to enlarge the grid or shorten the run, not evidence of a conservation bug.

In [ ]:
expected_volume = np.sum(influx) * esd.time_elapsed
final_volume = np.sum(esd.sediment_thickness) * dx * dy
np.isclose(final_volume, expected_volume, rtol=1e-6)

## Basement, sediment thickness, and topography

`GeoEnthalpyDelta` reads two forcing fields you supply (`sea_level__elevation` and `sediment__influx`) and updates one field, `topographic__elevation` $\eta = E + H$ (Eq. 2). The sediment thickness $H$ is available as `esd.sediment_thickness`, and the basement as their difference, $E = \eta - H$. Together they show,
node by node, how much bare basement lies below (1), how thick the mobile
deposit is (2), and what the combined land surface looks like relative to
sea level (3), the same quantities the paper's shoreline diagnostic
$\eta(x,y,t) = Z(t)$ (Sect. 3.3) would be computed from.

1. Basement elevation $E$

In [ ]:
basement = (grid.at_node["topographic__elevation"] - esd.sediment_thickness).reshape(
    grid.shape
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    basement,
    origin="lower",
    extent=(0, ncols * dx, 0, nrows * dy),
    cmap="viridis",
    aspect="equal",
)
ax.set_title("Basement elevation (heatmap)")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(im, ax=ax, label="elevation")
plt.show()

2. `esd.sediment_thickness`, the prognostic variable $H$ that the model actually integrates forward in time each step.

In [ ]:
Hthickness = esd.sediment_thickness.reshape(grid.shape)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    Hthickness,
    origin="lower",
    extent=(0, ncols * dx, 0, nrows * dy),
    cmap="viridis",
    aspect="equal",
)
ax.set_title("Sediment thickness")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(im, ax=ax, label="thickness")
plt.show()

3. `grid.at_node["topographic__elevation"]` the reconstructed surface $\eta = E + H$; compare it to the sea level line in the cross-section below to see where the shoreline currently sits.

In [ ]:
final_topography = grid.at_node["topographic__elevation"].reshape(grid.shape)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    final_topography,
    origin="lower",
    extent=(0, ncols * dx, 0, nrows * dy),
    cmap="terrain",
    aspect="equal",
)
ax.set_title("Final topography elevation")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(im, ax=ax, label="elevation")
plt.show()

## Comprehensive plots

The panels below combine the same fields into one figure: a map view of
$H$, a centerline cross-section comparing the reconstructed surface
$\eta$ against the fixed basement and current sea level, and the running
mass-balance check from above.

In [ ]:
H = Hthickness
eta = final_topography
final_sea_level = float(esd.sea_level)
X = grid.x_of_node.reshape(grid.shape)
Y = grid.y_of_node.reshape(grid.shape)
j_mid = nrows // 2

fig, axes = plt.subplots(3, 1, figsize=(5, 10))

im = axes[0].pcolormesh(X, Y, H, shading="auto", cmap="viridis")
axes[0].set_title("Sediment thickness")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_aspect("equal")
fig.colorbar(im, ax=axes[0], label="H")

axes[1].plot(X[j_mid], eta[j_mid], label="topography (basement + H)")
axes[1].plot(X[j_mid], basement[j_mid], "k--", label="basement")
axes[1].axhline(final_sea_level, color="tab:blue", linestyle=":", label="sea level")
axes[1].set_title("Centerline cross-section")
axes[1].set_xlabel("x")
axes[1].set_ylabel("elevation")
axes[1].legend(fontsize=8)

axes[2].plot(time_history, volume_history, label="modeled volume")
axes[2].plot(
    time_history,
    [np.sum(influx) * t for t in time_history],
    "k--",
    label="expected volume",
)
axes[2].set_title("Mass balance")
axes[2].set_xlabel("time")
axes[2].set_ylabel("total volume")
axes[2].legend(fontsize=8)

fig.tight_layout()

## Theory: what the code above is doing

### Delta evolution as a moving-boundary problem

Deltas grow (or retreat) by depositing sediment across a basin, and the geomorphic boundaries that define one, and their boundaries are not fixed in advance. They move as part of the solution, the same way a melting front advances in a classical Stefan (phase-change) problem.

`GeoEnthalpyDelta` Component implements the 2D fixed-grid model described in
Lorenzo-Trueba, Anderson, Bui, and Voller, *GeoEnthalpy-Delta v1.0: an enthalpy-based model for coupled subaerial and subaqueous delta evolution with diagnostic moving boundaries* (manuscript in preparation for
*Geoscientific Model Development*), which is able to create a model that can be used to tracks three boundaries:

- **ABT** (alluvial–bedrock transition): the upstream limit of finite
  sediment thickness.
- **SH** (shoreline): the contour where the reconstructed land surface
  crosses sea level.
- **TOE**: the offshore limit of finite sediment thickness.

Rather than tracking these boundaries explicitly, the model
evolves a single conserved field, sediment thickness $H$, on a **fixed**
grid, and *diagnoses* the boundaries afterward from $H$ and the
reconstructed surface. That fixed-grid, diagnose-it-afterward strategy is
the `enthalpy` in the name: it is borrowed from phase-change (Stefan)
problems in heat transfer, applied here to sediment mass instead of
thermal energy. The cells below work through the governing equations,
transport closure, and numerical scheme, and tie each one back to the
code you just ran.

### Governing equations

The primary state variable is the **sediment thickness above the fixed basement**, $H(x, y, t)$. The component tracks it internally rather than as a grid field; read it with the `sediment_thickness` property (e.g. `esd.sediment_thickness`).

Given the non-erodible basement elevation $E(x, y)$, the reconstructed land-surface elevation (`topographic__elevation`) is

$$
\eta(x, y, t) = E(x, y) + H(x, y, t).
$$

The basement is not a field either: at the start of every `run_one_step` call the component re-derives it as $E = \eta - H$. You therefore only supply the initial `topographic__elevation` (e.g. from a DEM), which should already include any initial sediment thickness passed in through the `sediment_thickness` keyword.

Sediment mass is conserved in flux form:

$$
\frac{\partial H}{\partial t} = -\nabla \cdot \mathbf{q} + \frac{S}{\Delta x\, \Delta y},
\qquad \mathbf{q} = (q_x, q_y),
$$

where $\mathbf{q}$ is the depth-integrated sediment flux and $S$ is the volumetric sediment supply at each node (the `sediment__influx` field). Because the
basement never changes, this is equivalent to evolving the land surface
directly, but writing it in terms of $H$ lets the model use $H = 0$ to
mean "no sediment yet," including at cells the delta hasn't reached —
there's no need to track a moving computational domain.

At every node, sea level $Z(t)$ (the `sea_level__elevation` grid field) splits the domain into two local transport
regimes, evaluated from the reconstructed surface:

$$
k(x,y,t) =
\begin{cases}
\text{topset}, & \eta(x,y,t) \ge Z(t) \\
\text{foreset}, & \eta(x,y,t) < Z(t)
\end{cases}
$$

so a node switches from topset to foreset (or back) automatically as $H$,
$\eta$, or $Z(t)$ evolve — the shoreline is simply wherever this switch
currently falls, rather than a boundary the code has to move by hand.

This is why **Grid setup** only creates `topographic__elevation`,
**External forcing** adds `sea_level__elevation` and `sediment__influx`,
and the run loop reads $H$ from `esd.sediment_thickness`.

### Critical slopes for sediment transport

Sediment is redistributed by a nonlinear diffusive flux driven by the
local slope of $\eta$, with a critical slope (threshold) below which no
transport occurs. The down-system (grid-x, feeder-to-basin) flux is
directed only seaward:

$$
q_x = D_k^x \max\!\left(-\frac{\partial \eta}{\partial x} - C_k^x,\; 0\right),
$$

while the lateral (grid-y) flux can act in either direction, down the
local cross-slope:

$$
q_y = -D_k^y \operatorname{sgn}\!\left(\frac{\partial \eta}{\partial y}\right)
\max\!\left(\left|\frac{\partial \eta}{\partial y}\right| - C_k^y,\; 0\right).
$$

The subscript $k \in \{\text{top}, \text{fore}\}$ means the diffusivity
$D_k$ and threshold slope $C_k$ switch depending on whether the *donor*
cell (the one sediment is leaving) is currently topset or foreset — so the
same flux law governs both the subaerial delta plain and the submerged
foreset, just with different mobility parameters in each regime.

Four independent parameter pairs therefore control transport: a
down-system and a lateral diffusivity/threshold pair for the topset, and
another pair for the foreset. In the component, each pair is a single keyword
that takes an `(x, y)` tuple (or a scalar, which applies to both directions):
- `topset_threshold=(C_top_x, C_top_y)`
- `foreset_threshold=(C_fore_x, C_fore_y)`
- `topset_diffusivity=(D_top_x, D_top_y)`
- `foreset_diffusivity=(D_fore_x, D_fore_y)`

These are the four keywords passed to `GeoEnthalpyDelta(...)` in
**Component setup**.

### Numerical Step `run_one_step`

`GeoEnthalpyDelta` solves the equations above with an explicit, conservative
finite-volume update on the fixed `RasterModelGrid`. Each substep (see `_advance_substep` in `geoenthalpy_delta.py`) first mixes the `sediment__influx` supply into $H$, so newly supplied sediment can move in that same substep, and then makes three passes:

1. **Candidate lateral fluxes** are computed from the cross-slope of $\eta$ at every internal grid-y face, using the donor cell's topset/foreset regime and its threshold/diffusivity.
2. **Lateral positivity limiter**: because sediment can leave a cell in both the $+y$ and $-y$ directions within the same step, candidate lateral fluxes are scaled down (never up) so that no cell can export more sediment than it currently holds.
3. **Downstream fluxes**, swept from the west (upstream) edge toward the east (downstream) edge: each cell's outgoing down-system flux is capped by both its transport capacity *and* the sediment actually available to it once its incoming upstream and already-limited lateral fluxes are accounted for.

This ordering guarantees $H \ge 0$ everywhere without an iterative solve,
at the cost of requiring an explicit, diffusion-type stable time step. You don't have to manage that yourself:
`run_one_step(dt)` splits any `dt` into as many substeps as the
Courant–Friedrichs–Lewy (CFL) condition requires (controlled by the `cfl` keyword, default 0.4). Calling `run_one_step()` with no `dt` takes a single stable substep.
That is why the run loop can use a fixed `dt = 0.1` without checking
stability.

**Boundary conditions.** Every node, including the grid's perimeter, is part of the transport domain, so `run_one_step` requires every node to have status `BC_NODE_IS_CORE` and raises a `ValueError` otherwise. No flux crosses the grid's outer edges: sediment only enters through `sediment__influx` and never leaves the domain. The deposit may therefore become artificially boundary-influenced if it approaches or reaches a lateral or downstream edge, even though the global sediment balance still closes. Mass conservation and boundary clearance should be checked separately.
This is why **Grid setup** sets every node to `BC_NODE_IS_CORE`, and why
the **Mass balance** check fails if the fan reaches the grid edges.